In [1]:
import importlib.util, sys

if importlib.util.find_spec("vosk") is None:
    !{sys.executable} -m pip install vosk

In [2]:
import os
import json
import wave
import numpy as np
from vosk import Model, KaldiRecognizer, SetLogLevel
from scipy.io import wavfile
from scipy.signal import butter, lfilter

# reduce Vosk log spam
SetLogLevel(-1)

# ---------- PROJECT PATHS ----------
# This assumes your notebook is inside: Ex3_files/
# and you have: Ex3_files/models/model-en , model-es , model-it
BASE_DIR = os.getcwd()

MODELS_PATHS = {
    "english":  os.path.join(BASE_DIR, "models", "model-en"),
    "italian":  os.path.join(BASE_DIR, "models", "model-it"),
    "spanish":  os.path.join(BASE_DIR, "models", "model-es"),
}

# Audio folders (recommended structure)
# Ex3_files/audio/EN , Ex3_files/audio/IT , Ex3_files/audio/ES
AUDIO_ROOT = os.path.join(BASE_DIR)
LANG_TO_FOLDER = {"english": "EN", "italian": "IT", "spanish": "ES"}

print("📂 BASE_DIR:", BASE_DIR)
print("📂 AUDIO_ROOT:", AUDIO_ROOT)
for k,v in MODELS_PATHS.items():
    print(f"Model {k}: {v} | exists={os.path.exists(v)}")


📂 BASE_DIR: /Users/syedabdullahhumayun/Desktop/final/isp abd finalF/Ex4_files
📂 AUDIO_ROOT: /Users/syedabdullahhumayun/Desktop/final/isp abd finalF/Ex4_files
Model english: /Users/syedabdullahhumayun/Desktop/final/isp abd finalF/Ex4_files/models/model-en | exists=True
Model italian: /Users/syedabdullahhumayun/Desktop/final/isp abd finalF/Ex4_files/models/model-it | exists=True
Model spanish: /Users/syedabdullahhumayun/Desktop/final/isp abd finalF/Ex4_files/models/model-es | exists=True


In [3]:
def bandpass_filter(x, sr, low=300, high=3400, order=4):
    # I use a simple Butterworth bandpass to keep mostly speech frequencies.
    # Voice is usually strongest around ~300–3400 Hz, so this trims rumble + harsh highs.
    nyq = 0.5 * sr
    lowc = low / nyq
    highc = min(high / nyq, 0.99)  # I clamp the high cutoff so it never hits/exceeds Nyquist

    # Butterworth is smooth/flat in the passband, which is good for “cleanup” filtering.
    b, a = butter(order, [lowc, highc], btype='band')
    return lfilter(b, a, x)

def preprocess_to_16k_mono_wav(in_wav, out_wav):
    # This is my “make it ASR-friendly” preprocessor:
    # - read WAV
    # - force mono
    # - normalize
    # - bandpass to speech range
    # - resample to 16 kHz (common ASR input rate)
    # - write back as int16 PCM
    sr, x = wavfile.read(in_wav)

    # Convert to float so filtering + interpolation behave nicely
    if x.dtype != np.float32:
        x = x.astype(np.float32)

    # If it’s stereo, I average channels into a single mono track
    if x.ndim == 2:
        x = x.mean(axis=1)

    # Normalize to prevent clipping and keep volume levels consistent across files
    mx = np.max(np.abs(x)) + 1e-9
    x = x / mx

    # Bandpass filter focused on speech energy (helps reduce noise outside voice band)
    x = bandpass_filter(x, sr)

    # Resample to 16 kHz using basic linear interpolation (fast and fine for a prototype)
    target_sr = 16000
    if sr != target_sr:
        # I map the signal to a normalized time axis (0..1) then interpolate onto the new grid
        t_old = np.linspace(0, 1, num=len(x), endpoint=False)
        t_new = np.linspace(0, 1, num=int(len(x) * target_sr / sr), endpoint=False)
        x = np.interp(t_new, t_old, x).astype(np.float32)
        sr = target_sr

    # Convert back to int16 PCM so it saves as a standard WAV that most tools accept
    x_i16 = np.clip(x * 32767.0, -32768, 32767).astype(np.int16)
    wavfile.write(out_wav, sr, x_i16)

    return out_wav


In [4]:
LOADED_MODELS = {}

def initialize_models():
    # I load each Vosk model once and keep it in memory, so I don’t reload it every time I run ASR.
    # MODELS_PATHS is expected to be a dict like: {"EN": ".../model-en", "ES": ".../model-es", ...}
    for lang, path in MODELS_PATHS.items():
        # If a model folder is missing, I stop immediately with a clear error message.
        if not os.path.exists(path):
            raise FileNotFoundError(f"Missing Vosk model folder for {lang}: {path}")

        # Model(path) loads the full ASR model from disk (this is the heavy step).
        LOADED_MODELS[lang] = Model(path)

    # Quick confirmation print so I can see what's available before running recognition.
    print("✅ Models loaded:", list(LOADED_MODELS.keys()))

initialize_models()


✅ Models loaded: ['english', 'italian', 'spanish']


In [5]:
def transcribe_audio(wav_path, lang):
    """
    wav_path must be a real WAV file (I get the best results with 16kHz mono).
    """
    # I grab the already-loaded model for the requested language.
    # This assumes initialize_models() has already run successfully.
    model = LOADED_MODELS[lang]

    # I open the WAV using Python’s built-in wave module so I can validate format quickly.
    with wave.open(wav_path, "rb") as wf:
        # Vosk/KaldiRecognizer expects mono audio (single channel).
        if wf.getnchannels() != 1:
            return "Error: WAV must be mono"

        # Sample width must be 16-bit PCM (2 bytes per sample).
        if wf.getsampwidth() != 2:
            return "Error: WAV must be 16-bit PCM"

        # I create the recognizer using the file’s sample rate.
        # 16k is ideal for most small Vosk models, but this will still run at other rates.
        rec = KaldiRecognizer(model, wf.getframerate())

        # I build the transcript in chunks so long files don’t blow up memory.
        text_out = []

        while True:
            # I read a fixed chunk of frames each loop (roughly a few hundred ms).
            data = wf.readframes(4000)
            if len(data) == 0:
                break

            # AcceptWaveform returns True when Vosk has a finalised segment result ready.
            if rec.AcceptWaveform(data):
                j = json.loads(rec.Result())
                if j.get("text"):
                    text_out.append(j["text"])

        # FinalResult flushes anything still buffered inside the recognizer.
        j = json.loads(rec.FinalResult())
        if j.get("text"):
            text_out.append(j["text"])

    # Join all recognised segments into one clean string.
    return " ".join(text_out).strip()


In [6]:
def wer_details(reference, hypothesis):
    # I compute Word Error Rate (WER) by comparing the reference transcript
    # against the ASR hypothesis transcript at the word level.
    #
    # WER = (Substitutions + Deletions + Insertions) / N * 100
    # where N is the number of words in the reference.

    # Lowercase + split keeps the scoring consistent (case doesn’t affect the metric).
    r = reference.lower().split()
    h = hypothesis.lower().split()

    # I build a classic edit-distance DP matrix:
    # d[i,j] = minimum edits to turn r[:i] into h[:j]
    d = np.zeros((len(r)+1, len(h)+1), dtype=np.int32)

    # op stores what operation produced the best score at each cell:
    # 0 = match (no error), 1 = substitution, 2 = insertion, 3 = deletion
    op = np.zeros((len(r)+1, len(h)+1), dtype=np.int8)

    # Base cases:
    # - If hypothesis is empty, I must delete i words
    for i in range(1, len(r)+1):
        d[i,0] = i
        op[i,0] = 3

    # - If reference is empty, I must insert j words
    for j in range(1, len(h)+1):
        d[0,j] = j
        op[0,j] = 2

    # Fill DP table
    for i in range(1, len(r)+1):
        for j in range(1, len(h)+1):
            # If the current words match, no extra cost (carry over diagonal).
            if r[i-1] == h[j-1]:
                d[i,j] = d[i-1,j-1]
                op[i,j] = 0
            else:
                # Otherwise I test the three possible edits:
                sub = d[i-1,j-1] + 1  # replace r[i-1] with h[j-1]
                ins = d[i, j-1] + 1   # insert h[j-1]
                dele = d[i-1, j] + 1  # delete r[i-1]

                # Pick the cheapest move
                best = min(sub, ins, dele)
                d[i,j] = best

                # Record which move I chose (used later for counting S/D/I)
                op[i,j] = 1 if best == sub else (2 if best == ins else 3)

    # Backtrack through op to count how many substitutions, deletions, and insertions happened.
    i, j = len(r), len(h)
    S = D = I = 0

    while i > 0 or j > 0:
        # Match: move diagonally with no error
        if i > 0 and j > 0 and op[i,j] == 0:
            i -= 1
            j -= 1

        # Substitution: move diagonally and count S
        elif i > 0 and j > 0 and op[i,j] == 1:
            S += 1
            i -= 1
            j -= 1

        # Insertion: move left and count I
        elif j > 0 and op[i,j] == 2:
            I += 1
            j -= 1

        # Deletion: move up and count D
        else:
            D += 1
            i -= 1

    # N is reference length (avoid divide-by-zero if reference is empty).
    N = max(len(r), 1)

    # Convert to percentage for readability in reports.
    wer = (S + D + I) / N * 100

    # I return both the WER % and the raw breakdown for reporting/debugging.
    return wer, (S, D, I, N)


In [7]:
# I keep these in one dictionary so my evaluation loop can automatically
# compute WER whenever a ground-truth sentence is available.

# key format: (lang, filename) -> reference text
REFERENCE = {
    ("english", "your_sentence1.wav"): "I love my pets",
    ("english", "your_sentence2.wav"): "Hello testing 1..2..3 this is a clear recording for my project",

    # ================= ENGLISH =================
    # These are the airport-style prompt sentences (adult + child versions).
    ("english", "checkin.wav"): "where is the check in desk",
    ("english", "checkin_child.wav"): "where is the check in desk",
    ("english", "parents.wav"): "i have lost my parents",
    ("english", "parents_child.wav"): "i have lost my parents",
    ("english", "suitcase.wav"): "please i have lost my suitcase",
    ("english", "suitcase_child.wav"): "please i have lost my suitcase",
    ("english", "what_time.wav"): "what time is my plane",
    ("english", "what_time_child.wav"): "what time is my plane",
    ("english", "where.wav"): "where are the restaurants and shops",
    ("english", "where_child.wav"): "where are the restaurants and shops",

    # ================= ITALIAN =================
    ("italian", "checkin_it.wav"): "dove è il bancone",
    ("italian", "parents_it.wav"): "ho perso i miei genitori",
    ("italian", "suitcase_it.wav"): "per favore ho perso la mia valigia",
    ("italian", "what_time_it.wav"): "a che ora è il mio aereo",
    ("italian", "where_it.wav"): "dove sono i ristoranti e i negozi",

    # ================= SPANISH =================
    ("spanish", "checkin_es.wav"): "dónde están los mostradores",
    ("spanish", "parents_es.wav"): "he perdido a mis padres",
    ("spanish", "suitcase_es.wav"): "por favor he perdido mi maleta",
    ("spanish", "what_time_es.wav"): "a qué hora es mi avión",
    ("spanish", "where_es.wav"): "dónde están los restaurantes y las tiendas",

    # If you add the provided client files later, I’d drop them in here like this:
    # ("english", "Ex4_sound1.wav"): "....",
}

def list_wavs_for_lang(lang):
    # I look up the folder for the requested language and list only .wav files.
    # This keeps the evaluation loop clean and avoids hardcoding filenames.
    folder = os.path.join(AUDIO_ROOT, LANG_TO_FOLDER[lang])
    if not os.path.exists(folder):
        return []
    return sorted([f for f in os.listdir(folder) if f.lower().endswith(".wav")])

def evaluate_language(lang):
    # This runs the whole pipeline for a single language:
    # 1) list wavs
    # 2) preprocess each one (mono + 16k + bandpass)
    # 3) run Vosk transcription
    # 4) compute WER when a reference sentence exists
    folder = os.path.join(AUDIO_ROOT, LANG_TO_FOLDER[lang])
    wavs = list_wavs_for_lang(lang)

    print(f"\n=== {lang.upper()} ({folder}) ===")
    if not wavs:
        print("No .wav files found.")
        return

    # Quick table header so results are easy to scan during marking/demo.
    print(f"{'File':<28} | {'Hypothesis':<40} | {'WER%':>6} | S D I N")
    print("-"*95)

    for fname in wavs:
        in_path = os.path.join(folder, fname)

        # I write a temporary processed file next to the original.
        # (I delete it after transcription so the folder doesn’t fill up.)
        proc_path = os.path.join(folder, "processed_" + fname)

        # --- Preprocess for ASR ---
        preprocess_to_16k_mono_wav(in_path, proc_path)

        # --- ASR ---
        hyp = transcribe_audio(proc_path, lang)

        # --- Cleanup ---
        # I remove the processed file so each run stays repeatable and tidy.
        if os.path.exists(proc_path):
            os.remove(proc_path)

        # --- Scoring (WER) ---
        # I only compute WER when I have a reference sentence for this exact (lang, file).
        ref = REFERENCE.get((lang, fname))

        # If there’s no reference or the recognizer returned a format error, I skip WER.
        if ref is None or hyp.startswith("Error"):
            print(f"{fname:<28} | {hyp[:40]:<40} | {'---':>6} | - - - -")
        else:
            # I compute the WER breakdown so I can report substitutions/deletions/insertions too.
            wer, (S, D, I, N) = wer_details(ref, hyp)

            # I shorten long hypotheses so the table stays aligned.
            display = (hyp[:37] + "...") if len(hyp) > 40 else hyp
            print(f"{fname:<28} | {display:<40} | {wer:>6.1f} | {S} {D} {I} {N}")

# Run all languages in one go so I get a full summary quickly.
for lang in ["english", "italian", "spanish"]:
    evaluate_language(lang)



=== ENGLISH (/Users/syedabdullahhumayun/Desktop/final/isp abd finalF/Ex4_files/EN) ===
File                         | Hypothesis                               |   WER% | S D I N
-----------------------------------------------------------------------------------------------
checkin.wav                  | where is the check in desk               |    0.0 | 0 0 0 6
parents.wav                  | i lost my therapist                      |   40.0 | 1 1 0 5
suitcase.wav                 | police have lost my suitcase             |   33.3 | 1 1 0 6
what_time.wav                | what time is my life                     |   20.0 | 1 0 0 5
where.wav                    | where are the restaurants and shops      |    0.0 | 0 0 0 6
your_sentence1.wav           | i love my bags                           |   25.0 | 1 0 0 4
your_sentence2.wav           | hello testing one two three this is a... |   36.4 | 2 0 2 11

=== ITALIAN (/Users/syedabdullahhumayun/Desktop/final/isp abd finalF/Ex4_files/IT) ===

In [8]:
import csv

def run_all_and_save_csv(out_csv="ex4_results.csv"):
    # This is my “batch runner” for Exercise 4:
    # I loop through every language folder, run preprocessing + ASR, compute WER when possible,
    # and then save everything into a single CSV so it’s easy to include in the report.
    rows = []

    for lang in ["english", "italian", "spanish"]:
        # I resolve the folder path for the language (based on your AUDIO_ROOT + LANG_TO_FOLDER map).
        folder = os.path.join(AUDIO_ROOT, LANG_TO_FOLDER[lang])

        # I only pick .wav files so random files in the folder don’t break the run.
        wavs = sorted([f for f in os.listdir(folder) if f.lower().endswith(".wav")])

        for fname in wavs:
            in_wav = os.path.join(folder, fname)

            # I create a temporary “processed” WAV for ASR (mono + 16k + bandpass).
            # I delete it after recognition so the folder stays clean.
            proc_wav = os.path.join(folder, "proc_" + fname)

            # --- Preprocess (ASR-friendly format) ---
            preprocess_to_16k_mono_wav(in_wav, proc_wav)

            # --- Transcribe using the loaded Vosk model for that language ---
            hyp = transcribe_audio(proc_wav, lang)

            # --- Cleanup temp file ---
            if os.path.exists(proc_wav):
                os.remove(proc_wav)

            # --- Scoring ---
            # If I have a reference for this exact file, I compute WER and store the breakdown.
            ref = REFERENCE.get((lang, fname))

            if ref is None or hyp.startswith("Error"):
                # If there’s no reference (or ASR returned a format error), I still log it,
                # but I leave WER fields blank so the CSV stays consistent.
                rows.append([lang, fname, ref if ref else "", hyp, "", "", "", "", ""])
            else:
                wer, (S, D, I, N) = wer_details(ref, hyp)

                # I round WER to 1 decimal for neatness in the spreadsheet/report.
                rows.append([lang, fname, ref, hyp, round(wer, 1), S, D, I, N])

    # Write the final CSV with a clear header row.
    with open(out_csv, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["Language", "File", "Reference", "Hypothesis", "WER_%", "S", "D", "I", "N"])
        w.writerows(rows)

    print("Saved:", out_csv)

run_all_and_save_csv("ex4_results.csv")


Saved: ex4_results.csv
